# examples-seen-step-axis — ex3: gradient-accumulation factor in examples_seen

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `examples-seen-step-axis`. Running the final beacon cell reports progress against the `Trainer: examples-seen step axis` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Trainer: examples-seen step axis` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`examples-seen-step-axis`** (exercise 3). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "examples-seen-step-axis"
DD_SUBTOPIC = "Trainer: examples-seen step axis"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Gradient accumulation — examples_seen = step * batch_size * accum_steps

Ex1 + ex2 used `examples_seen = step * batch_size`. The deepening move handles gradient accumulation: you do `accum_steps` forward+backward passes BEFORE calling `optimizer.step()`. From the optimizer's view each 'step' actually saw `accum_steps` micro-batches.

```python
# step = optimizer steps. micro-batch = backward but no step yet.
examples_seen = step * batch_size * accum_steps
```

**Why this matters.** You want curves COMPARABLE across runs with different `batch_size` AND different `accum_steps`. Two runs with `(B=8, accum=4)` and `(B=32, accum=1)` see the same effective batch (32 examples per optimizer step), so logging by `examples_seen` makes their loss curves overlap. Logging by `step` would make them diverge visually for no real reason.

**`accum_steps=1` collapses to ex1/ex2.** When you're not accumulating, `step * batch_size * 1 == step * batch_size`. The formula is a strict generalization — same shape, extra factor.

**It's about effective examples, not micro-batches.** The micro-batch count INSIDE one optimizer step is `accum_steps`. The optimizer step counter increments only AFTER all micro-batches have backprop'd. So `examples_seen` per optimizer step grows by `batch_size * accum_steps`.

### Exercise 3 — gradient-accumulation factor in examples_seen

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Apply
> LO: Apply `examples_seen = step * batch_size * accum_steps` so that training runs with different micro-batch / accumulation splits but the same EFFECTIVE batch size produce overlapping wandb curves.
> Keywords: examples-seen, gradient-accumulation, effective-batch, wandb
> ```

**KCs targeted:** `examples-seen-times-accum-steps-factor`, `optimizer-step-counter-separate-from-micro-batch`

Implement `ex3_train_with_grad_accum(losses, batch_size, accum_steps, wandb)`. A training-loop-like driver that logs one `wandb.log(...)` per OPTIMIZER step, where each step represents `accum_steps` micro-batches.

Inputs:
- `losses`: `list[float]` — one entry per OPTIMIZER step (already averaged across the micro-batches inside that step).
- `batch_size`: int — micro-batch size.
- `accum_steps`: int — number of micro-batches per optimizer step.
- `wandb`: object with `wandb.log(dict)`.

For `step, loss in enumerate(losses, start=1)`:

1. Compute `examples_seen = step * batch_size * accum_steps`.
2. Call `wandb.log({'loss': loss, 'examples_seen': examples_seen, 'step': step})`.

Return the total number of log calls made (== `len(losses)`).

In [ ]:
def ex3_train_with_grad_accum(losses, batch_size, accum_steps, wandb) -> int:
    """One wandb.log per optimizer step; examples_seen includes accum_steps factor."""
    raise NotImplementedError()


def _test_ex3():
    from unittest.mock import MagicMock

    # === Basic case: 5 optimizer steps, batch_size=8, accum_steps=4 ===
    # effective batch = 32; per step examples_seen = 32, 64, 96, 128, 160.
    wandb = MagicMock()
    losses = [1.0, 0.8, 0.7, 0.65, 0.6]
    n = ex3_train_with_grad_accum(losses, batch_size=8, accum_steps=4, wandb=wandb)
    assert n == 5, f'expected 5 log calls; got {n}'
    assert wandb.log.call_count == 5

    for i, call in enumerate(wandb.log.call_args_list, start=1):
        args, kwargs = call
        assert len(args) == 1, f'call {i}: expected one positional dict; got {args}'
        payload = args[0]
        assert isinstance(payload, dict)
        assert payload['examples_seen'] == i * 8 * 4, (
            f'call {i}: examples_seen should be {i*32}; got {payload["examples_seen"]}'
        )
        assert payload['step'] == i, f'call {i}: step should be {i}; got {payload["step"]}'
        assert payload['loss'] == losses[i - 1]

    # === accum_steps=1 collapses to ex1/ex2 behaviour ===
    wandb2 = MagicMock()
    ex3_train_with_grad_accum([0.5, 0.4, 0.3], batch_size=32, accum_steps=1, wandb=wandb2)
    payloads = [c.args[0] for c in wandb2.log.call_args_list]
    assert payloads[0]['examples_seen'] == 32
    assert payloads[1]['examples_seen'] == 64
    assert payloads[2]['examples_seen'] == 96

    # === Two runs with same effective batch overlap ===
    # Run A: bs=8, accum=4 → effective 32
    # Run B: bs=32, accum=1 → effective 32
    # After K optimizer steps, examples_seen should be identical.
    wandb_a = MagicMock()
    wandb_b = MagicMock()
    ex3_train_with_grad_accum([0.1] * 10, batch_size=8, accum_steps=4, wandb=wandb_a)
    ex3_train_with_grad_accum([0.1] * 10, batch_size=32, accum_steps=1, wandb=wandb_b)
    a_examples = [c.args[0]['examples_seen'] for c in wandb_a.log.call_args_list]
    b_examples = [c.args[0]['examples_seen'] for c in wandb_b.log.call_args_list]
    assert a_examples == b_examples, (
        f'runs with same effective batch must agree on examples_seen:\n  '
        f'A={a_examples}\n  B={b_examples}'
    )

    # === accum_steps=8 multiplies examples_seen 8x relative to accum=1 ===
    wandb_x = MagicMock()
    wandb_y = MagicMock()
    ex3_train_with_grad_accum([0.1] * 3, batch_size=16, accum_steps=1, wandb=wandb_x)
    ex3_train_with_grad_accum([0.1] * 3, batch_size=16, accum_steps=8, wandb=wandb_y)
    x_ex = [c.args[0]['examples_seen'] for c in wandb_x.log.call_args_list]
    y_ex = [c.args[0]['examples_seen'] for c in wandb_y.log.call_args_list]
    for xi, yi in zip(x_ex, y_ex):
        assert yi == xi * 8, f'accum=8 should be 8x the accum=1 examples_seen; got {yi} vs {xi}'

    # === Empty losses → zero calls ===
    wandb3 = MagicMock()
    n0 = ex3_train_with_grad_accum([], batch_size=16, accum_steps=2, wandb=wandb3)
    assert n0 == 0
    assert wandb3.log.call_count == 0

    # === Payload key check ===
    wandb4 = MagicMock()
    ex3_train_with_grad_accum([0.9], batch_size=4, accum_steps=2, wandb=wandb4)
    payload = wandb4.log.call_args_list[0].args[0]
    assert set(payload.keys()) >= {'loss', 'examples_seen', 'step'}, (
        f'payload must contain loss, examples_seen, step; got keys {set(payload.keys())}'
    )
    _dd_passed.add('ex3')
    print("ex3 ✓")

_test_ex3()

<details><summary>Solution</summary>

```python
def ex3_train_with_grad_accum(losses, batch_size, accum_steps, wandb):
    n_calls = 0
    for step, loss in enumerate(losses, start=1):
        examples_seen = step * batch_size * accum_steps
        wandb.log({
            'loss': loss,
            'examples_seen': examples_seen,
            'step': step,
        })
        n_calls += 1
    return n_calls
```

**Effective batch is what curves are compared on.** Two runs with `(bs=8, accum=4)` and `(bs=32, accum=1)` both have effective batch 32 per optimizer step. `examples_seen` puts them on the same x-axis — their loss curves should look identical (up to randomness in mini-batch shuffling).

**`accum_steps=1` is the no-accumulation case.** The formula reduces to ex1/ex2's `step * batch_size` because the third factor is just 1. This is why the extension is non-breaking.

**Why log `step` AND `examples_seen`.** They serve different questions: `step` answers 'how many optimizer updates?' (cost on the optimizer's side); `examples_seen` answers 'how much data has the model been trained on?' (the comparison axis across runs).
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex3'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex3',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()